# 05 — Reprodukcja statystyk, tabel i wykresów z pracy magisterskiej

**Cel.** Centralny notebook, który prezentuje *wszystkie* artefakty cytowane
w pracy magisterskiej (tabele T1–T21, wykresy W1–W18) — zgodnie z
`appendix/INDEX.md`. Pliki źródłowe znajdują się w `appendix/`; notebook
renderuje je w czytelnej, jednolitej formie.

**Charakter notebooka.** Nie generuje wyników od zera. Jego rolą jest
*zaprezentowanie* gotowych artefaktów (CSV, TeX, PDF, PNG) oraz pokazanie
kolejności, w jakiej zostały one zacytowane w pracy. Dzięki temu recenzent
może w jednym miejscu prześledzić cały materiał empiryczny pracy.

**Zawartość:**
1. Tabela 1 — budżet obliczeniowy offline.
2. Tabele T2–T21 — statystyki opisowe, Friedman + A12, Wilson 95% CI.
3. Wykresy W1–W18 — bar/box/konwergencja (PDF + PNG w `appendix/C_plots/`).
4. Panele zbiorcze `thesis_stat_tables` (gotowe panele PNG do pracy).

In [ ]:
import prepare_notebook  # noqa: F401

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path(prepare_notebook.project_root)
APPENDIX = PROJECT_ROOT / "appendix"
PLOTS = APPENDIX / "C_plots"
STATS = APPENDIX / "B_statistical_tests"
THESIS_PANELS = STATS / "thesis_stat_tables"
assert PLOTS.is_dir() and STATS.is_dir()

# Wyświetlanie pełnych szerokości kolumn (długie nazwy run_id w CSV).
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## T1 — Tabela 1: budżet obliczeniowy offline

Źródło: [`appendix/B_statistical_tests/budget_table.md`](../appendix/B_statistical_tests/budget_table.md).
Wartości policzone ze wzoru `P × G × K × D` na podstawie YAML-i z
`configs/optimizer/`.

In [ ]:
display(Markdown((STATS / "budget_table.md").read_text(encoding="utf-8")))

## T2–T21 — statystyki opisowe, Friedman+A12, Wilson 95% CI

Mapowanie (z `INDEX.md`):

* Statystyki opisowe → `summary/summary_<metric>.csv` (offline) lub
  `summary/online_summary_<metric>.csv` (online).
* Friedman+CD → `friedman/{forest|urban}_friedman_<metric>.csv`.
* A12 → `a12/{forest|urban}_a12_<metric>.csv`.
* Wilson → `wilson/{failure_rate_online|failure_rate_offline|evasion_success_rate}.csv`.

In [ ]:
# Pełna mapa metryk i odpowiadających plików.
TABLE_MANIFEST = [
    # (kod, opis, paths…)
    ("T2", "Statystyki — bezpieczeństwo trajektorii (F3+F5)",
        STATS / "summary/summary_trajectory_safety_f3_f5.csv"),
    ("T3", "Friedman + A12 — bezpieczeństwo trajektorii",
        STATS / "friedman/forest_friedman_trajectory_safety_f3_f5.csv",
        STATS / "friedman/urban_friedman_trajectory_safety_f3_f5.csv",
        STATS / "a12/forest_a12_trajectory_safety_f3_f5.csv",
        STATS / "a12/urban_a12_trajectory_safety_f3_f5.csv"),
    ("T4", "Statystyki — długość trajektorii (F1)",
        STATS / "summary/summary_trajectory_length_f1.csv"),
    ("T5", "Friedman + A12 — długość trajektorii",
        STATS / "friedman/forest_friedman_trajectory_length_f1.csv",
        STATS / "friedman/urban_friedman_trajectory_length_f1.csv",
        STATS / "a12/forest_a12_trajectory_length_f1.csv",
        STATS / "a12/urban_a12_trajectory_length_f1.csv"),
    ("T6", "Statystyki — gładkość (F2+F4)",
        STATS / "summary/summary_trajectory_smoothness_f2_f4.csv"),
    ("T7", "Friedman + A12 — gładkość",
        STATS / "friedman/forest_friedman_trajectory_smoothness_f2_f4.csv",
        STATS / "friedman/urban_friedman_trajectory_smoothness_f2_f4.csv",
        STATS / "a12/forest_a12_trajectory_smoothness_f2_f4.csv",
        STATS / "a12/urban_a12_trajectory_smoothness_f2_f4.csv"),
    ("T8", "Statystyki — spójność roju",
        STATS / "summary/summary_swarm_cohesion_deviation.csv"),
    ("T9", "Friedman + A12 — spójność roju",
        STATS / "friedman/forest_friedman_swarm_cohesion_deviation.csv",
        STATS / "friedman/urban_friedman_swarm_cohesion_deviation.csv",
        STATS / "a12/forest_a12_swarm_cohesion_deviation.csv",
        STATS / "a12/urban_a12_swarm_cohesion_deviation.csv"),
    ("T10", "Statystyki — długość krzywej unikowej",
        STATS / "summary/online_summary_mean_evasion_arc_length_m.csv"),
    ("T11", "Friedman + A12 — długość krzywej unikowej",
        STATS / "friedman/online_forest_friedman_mean_evasion_arc_length_m.csv",
        STATS / "friedman/online_urban_friedman_mean_evasion_arc_length_m.csv",
        STATS / "a12/online_forest_a12_mean_evasion_arc_length_m.csv",
        STATS / "a12/online_urban_a12_mean_evasion_arc_length_m.csv"),
    ("T12", "Statystyki — skuteczność powrotu na trajektorię",
        STATS / "summary/online_summary_rejoin_quality.csv"),
    ("T13", "Friedman + A12 — skuteczność powrotu",
        STATS / "friedman/online_forest_friedman_rejoin_quality.csv",
        STATS / "friedman/online_urban_friedman_rejoin_quality.csv",
        STATS / "a12/online_forest_a12_rejoin_quality.csv",
        STATS / "a12/online_urban_a12_rejoin_quality.csv"),
    ("T14", "Statystyki — stosunek udanych uników (Wilson)",
        STATS / "wilson/evasion_success_rate.csv"),
    ("T15", "Wilson 95% CI — bezpieczeństwo uników",
        STATS / "wilson/failure_rate_online.csv",
        STATS / "wilson/failure_rate_offline.csv"),
    ("T16", "Statystyki — wartość optymalizacji offline",
        STATS / "summary/summary_final_objective.csv"),
    ("T17", "Friedman + A12 — skuteczność optymalizacji offline",
        STATS / "friedman/forest_friedman_final_objective.csv",
        STATS / "friedman/urban_friedman_final_objective.csv",
        STATS / "a12/forest_a12_final_objective.csv",
        STATS / "a12/urban_a12_final_objective.csv"),
    ("T18", "Friedman + A12 — szybkość optymalizacji offline (AUC)",
        STATS / "friedman/forest_friedman_auc_best_so_far.csv",
        STATS / "friedman/urban_friedman_auc_best_so_far.csv",
        STATS / "a12/forest_a12_auc_best_so_far.csv",
        STATS / "a12/urban_a12_auc_best_so_far.csv"),
    ("T19", "Statystyki — wartość optymalizacji online",
        STATS / "summary/online_summary_mean_online_best_fitness.csv"),
    ("T20", "Friedman + A12 — optymalizacja online",
        STATS / "friedman/online_forest_friedman_mean_online_best_fitness.csv",
        STATS / "friedman/online_urban_friedman_mean_online_best_fitness.csv",
        STATS / "a12/online_forest_a12_mean_online_best_fitness.csv",
        STATS / "a12/online_urban_a12_mean_online_best_fitness.csv"),
    ("T21", "Friedman + A12 — SP1 online",
        STATS / "friedman/online_forest_friedman_online_sp1.csv",
        STATS / "friedman/online_urban_friedman_online_sp1.csv",
        STATS / "a12/online_forest_a12_online_sp1.csv",
        STATS / "a12/online_urban_a12_online_sp1.csv"),
]

def render_table(code: str, desc: str, *paths: Path) -> None:
    display(Markdown(f"### {code} — {desc}"))
    for p in paths:
        if not p.exists():
            display(Markdown(f"*Brak pliku: `{p.relative_to(PROJECT_ROOT)}`*"))
            continue
        display(Markdown(f"`{p.relative_to(PROJECT_ROOT)}`"))
        display(pd.read_csv(p))

for entry in TABLE_MANIFEST:
    render_table(entry[0], entry[1], *entry[2:])

## W1–W18 — wykresy bar/box/konwergencja

Mapowanie z `INDEX.md`, sekcja B. Każdy wykres istnieje w wariantach
PDF (do druku) i PNG (do widoku notebookowego).

In [ ]:
PLOT_MANIFEST = [
    ("W1",  "Bar: odsetek trajektorii kolizyjnych, forest",
        PLOTS / "bar/bar_forest_failure_rate_offline.png"),
    ("W2",  "Bar: odsetek trajektorii kolizyjnych, urban",
        PLOTS / "bar/bar_urban_failure_rate_offline.png"),
    ("W3",  "Box: długość trajektorii F1, forest",
        PLOTS / "boxplots/boxplot_forest_trajectory_length_f1.png"),
    ("W4",  "Box: długość trajektorii F1, urban",
        PLOTS / "boxplots/boxplot_urban_trajectory_length_f1.png"),
    ("W5",  "Box: gładkość F2+F4, forest",
        PLOTS / "boxplots/boxplot_forest_trajectory_smoothness_f2_f4.png"),
    ("W6",  "Box: gładkość F2+F4, urban",
        PLOTS / "boxplots/boxplot_urban_trajectory_smoothness_f2_f4.png"),
    ("W7",  "Box: spójność roju, forest",
        PLOTS / "boxplots/boxplot_forest_swarm_cohesion_deviation.png"),
    ("W8",  "Box: spójność roju, urban",
        PLOTS / "boxplots/boxplot_urban_swarm_cohesion_deviation.png"),
    ("W9",  "Box: długość krzywej unikowej, forest",
        PLOTS / "boxplots/boxplot_forest_mean_evasion_arc_length_m.png"),
    ("W10", "Box: długość krzywej unikowej, urban",
        PLOTS / "boxplots/boxplot_urban_mean_evasion_arc_length_m.png"),
    ("W11", "Box: skuteczność powrotu, forest",
        PLOTS / "boxplots/boxplot_forest_rejoin_quality.png"),
    ("W12", "Box: skuteczność powrotu, urban",
        PLOTS / "boxplots/boxplot_urban_rejoin_quality.png"),
    ("W13", "Box: wartość optymalizacji, forest",
        PLOTS / "boxplots/boxplot_forest_final_objective.png"),
    ("W14", "Box: wartość optymalizacji, urban",
        PLOTS / "boxplots/boxplot_urban_final_objective.png"),
    ("W15", "Line: tempo optymalizacji offline, forest",
        PLOTS / "convergence/convergence_forest_best_so_far.png"),
    ("W16", "Line: tempo optymalizacji offline, urban",
        PLOTS / "convergence/convergence_urban_best_so_far.png"),
    ("W17", "Line: tempo optymalizacji online, forest",
        PLOTS / "convergence/online_convergence_forest.png"),
    ("W18", "Line: tempo optymalizacji online, urban",
        PLOTS / "convergence/online_convergence_urban.png"),
]

for code, desc, path in PLOT_MANIFEST:
    display(Markdown(f"### {code} — {desc}"))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"*Brak pliku: `{path.relative_to(PROJECT_ROOT)}`*"))

## Panele zbiorcze (`thesis_stat_tables/`)

Gotowe panele PNG generowane przez `scripts/generate_thesis_stat_tables.py`
— używane bezpośrednio w pracy magisterskiej (rozdziały 3.2.1.1, 3.2.1.2,
3.2.2.1, 3.2.2.2). Łączą Friedman + A12 + side-by-side forest/urban
dla każdej metryki w jednej grafice.

In [ ]:
for p in sorted(THESIS_PANELS.glob("*.png")):
    display(Markdown(f"### {p.stem}"))
    display(Image(filename=str(p)))

---
## Dalsze artefakty

* **Schemat bazy danych analytycznej:** `appendix/D_database_schema/`.
* **Konfiguracje Hydry (per algorytm × środowisko):** `appendix/E_configs/`.
* **Snapshot środowiska conda:** `appendix/F_environment/`.
* **Manifest 240 runów:** `appendix/H_run_manifest.csv`.

Pełny opis: [`appendix/INDEX.md`](../appendix/INDEX.md).